# Лабораторна робота №2. Частина 2
## Наука про дані: Individual Household Electric Power Consumption

### Імпорт бібліотек

In [1]:
import os
import zipfile
import urllib.request
import pandas as pd
import numpy as np
import timeit
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler

print("Бібліотеки імпортовано успішно.")

Бібліотеки імпортовано успішно.


### Завантаження та читання датасету
Індивідуальне споживання електроенергії домогосподарства (UCI ML Repository).

In [2]:
DATA_URL = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
ZIP_FILE = "household_power_consumption.zip"
CSV_FILE = "household_power_consumption.txt"

if not os.path.exists(CSV_FILE):
    if not os.path.exists(ZIP_FILE):
        print("Завантаження датасету...")
        urllib.request.urlretrieve(DATA_URL, ZIP_FILE)
        print("Завантажено.")
    
    print("Розпакування...")
    with zipfile.ZipFile(ZIP_FILE, "r") as z:
        z.extractall(".")
    print("Розпаковано.")
else:
    print(f"Файл {CSV_FILE} вже існує.")

df = pd.read_csv(
    CSV_FILE, sep=";", na_values=["?"],
    low_memory=False
)
print(f"Розмір: {df.shape}")
df.head()

Файл household_power_consumption.txt вже існує.
Розмір: (2075259, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


### Data Cleaning
Очищення даних: обробка пропусків, приведення типів, створення datetime індексу.

In [3]:
print("Пропуски до очищення:")
print(df.isnull().sum())
print(f"\nВідсоток пропусків: {df.isnull().sum().sum() / df.size * 100:.2f}%")

# Привести числові стовпці до numeric
numeric_cols = ["Global_active_power", "Global_reactive_power", "Voltage",
                "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Створити datetime стовпець
df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H:%M:%S")

# Заповнити пропуски медіаною
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Видалити рядки, де Date/Time не вдалося розпарсити
df = df.dropna(subset=["Datetime"])

# Додати допоміжні стовпці
df["Hour"] = df["Datetime"].dt.hour
df["DayOfWeek"] = df["Datetime"].dt.dayofweek
df["Month"] = df["Datetime"].dt.month

print(f"\nРозмір після очищення: {df.shape}")
print(f"Пропуски після очищення: {df.isnull().sum().sum()}")
df.dtypes

Пропуски до очищення:
Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64

Відсоток пропусків: 0.97%

Розмір після очищення: (2075259, 13)
Пропуски після очищення: 0


Date                             object
Time                             object
Global_active_power             float64
Global_reactive_power           float64
Voltage                         float64
Global_intensity                float64
Sub_metering_1                  float64
Sub_metering_2                  float64
Sub_metering_3                  float64
Datetime                 datetime64[ns]
Hour                              int32
DayOfWeek                         int32
Month                             int32
dtype: object

### Завдання 1: Записи з Global_active_power > 5 кВт
Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [4]:
def filter_high_power(df, threshold=5.0):
    """Обрати записи з Global_active_power > threshold кВт."""
    return df[df["Global_active_power"] > threshold].copy()

t = timeit.timeit(lambda: filter_high_power(df), number=10)
print(f"Час виконання (10 разів): {t:.4f} с")

high_power = filter_high_power(df)
print(f"\nЗаписів з потужністю > 5 кВт: {len(high_power)} ({len(high_power)/len(df)*100:.2f}%)")
print(f"Середня потужність у вибірці: {high_power['Global_active_power'].mean():.2f} кВт")
high_power.head(10)

Час виконання (10 разів): 0.0387 с

Записів з потужністю > 5 кВт: 17547 (0.85%)
Середня потужність у вибірці: 5.85 кВт


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime,Hour,DayOfWeek,Month
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,17,5,12
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,17,5,12
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,17,5,12
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00,17,5,12
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00,17,5,12
13,16/12/2006,17:37:00,5.268,0.398,232.91,22.6,0.0,2.0,17.0,2006-12-16 17:37:00,17,5,12
20,16/12/2006,17:44:00,5.894,0.000,232.69,25.4,0.0,0.0,16.0,2006-12-16 17:44:00,17,5,12
21,16/12/2006,17:45:00,7.706,0.000,230.98,33.2,0.0,0.0,17.0,2006-12-16 17:45:00,17,5,12
22,16/12/2006,17:46:00,7.026,0.000,232.21,30.6,0.0,0.0,16.0,2006-12-16 17:46:00,17,5,12
23,16/12/2006,17:47:00,5.174,0.000,234.19,22.0,0.0,0.0,17.0,2006-12-16 17:47:00,17,5,12


### Завдання 2: Сила струму 19–20 А, пральна+холодильник > бойлер+кондиціонер
Обрати записи з Global_intensity в межах 19–20 А, серед них — ті, де Sub_metering_2 (пральна, сушарка, холодильник, освітлення) > Sub_metering_3 (бойлер, кондиціонер).

In [5]:
def filter_current_and_metering(df):
    """Записи з силою струму 19-20 А, де Sub_metering_2 > Sub_metering_3."""
    current_mask = (df["Global_intensity"] >= 19) & (df["Global_intensity"] <= 20)
    current_df = df[current_mask]
    result = current_df[current_df["Sub_metering_2"] > current_df["Sub_metering_3"]]
    return result.copy()

t = timeit.timeit(lambda: filter_current_and_metering(df), number=10)
print(f"Час виконання (10 разів): {t:.4f} с")

current_filtered = filter_current_and_metering(df)
print(f"\nЗаписів з силою струму 19-20 А: {len(df[(df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20)])}")
print(f"З них Sub_metering_2 > Sub_metering_3: {len(current_filtered)}")
current_filtered.head(10)

Час виконання (10 разів): 0.0456 с

Записів з силою струму 19-20 А: 7021
З них Sub_metering_2 > Sub_metering_3: 2509


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime,Hour,DayOfWeek,Month
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00,18,5,12
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00,1,6,12
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00,1,6,12
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00,1,6,12
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00,1,6,12
477,17/12/2006,01:21:00,4.652,0.142,237.92,19.4,0.0,36.0,0.0,2006-12-17 01:21:00,1,6,12
508,17/12/2006,01:52:00,4.622,0.240,239.59,19.2,0.0,37.0,0.0,2006-12-17 01:52:00,1,6,12
944,17/12/2006,09:08:00,4.762,0.088,237.44,20.0,0.0,38.0,0.0,2006-12-17 09:08:00,9,6,12
945,17/12/2006,09:09:00,4.506,0.088,237.25,19.0,0.0,38.0,0.0,2006-12-17 09:09:00,9,6,12
1051,17/12/2006,10:55:00,4.444,0.136,235.97,19.2,2.0,50.0,17.0,2006-12-17 10:55:00,10,6,12


### Завдання 3: Випадкова вибірка 500 000 записів
Обрати випадковим чином 500 000 записів (без повторів), обчислити середні величини трьох груп споживання.

In [6]:
def random_sample_means(df, n=500000, seed=42):
    """Випадкова вибірка n записів, середні 3 груп споживання."""
    sample = df.sample(n=n, replace=False, random_state=seed)
    means = {
        "Sub_metering_1 (кухня)": sample["Sub_metering_1"].mean(),
        "Sub_metering_2 (пральня)": sample["Sub_metering_2"].mean(),
        "Sub_metering_3 (бойлер+конд.)": sample["Sub_metering_3"].mean(),
    }
    return sample, means

t = timeit.timeit(lambda: random_sample_means(df), number=10)
print(f"Час виконання (10 разів): {t:.4f} с")

sample, means = random_sample_means(df)
print(f"\nРозмір вибірки: {len(sample)}")
print("\nСередні величини споживання (Вт·год):")
for name, val in means.items():
    print(f"  {name}: {val:.4f}")

Час виконання (10 разів): 0.6982 с

Розмір вибірки: 500000

Середні величини споживання (Вт·год):
  Sub_metering_1 (кухня): 1.1090
  Sub_metering_2 (пральня): 1.2837
  Sub_metering_3 (бойлер+конд.): 6.3971


### Завдання 4: Записи після 18:00 з потужністю > 6 кВт
Обрати записи після 18:00, де середня потужність > 6 кВт за хвилину. Серед них — де група 2 (Sub_metering_2) є найбільшою. Потім: кожен 3-й з першої половини та кожен 4-й з другої половини.

In [7]:
def filter_evening_high_power(df):
    """
    1) Записи після 18:00 з Global_active_power > 6 кВт
    2) Серед них — де Sub_metering_2 найбільша
    3) Кожен 3-й з першої половини, кожен 4-й з другої
    """
    # Крок 1: після 18:00, потужність > 6 кВт
    mask = (df["Hour"] >= 18) & (df["Global_active_power"] > 6)
    evening = df[mask].copy()
    
    # Крок 2: Sub_metering_2 найбільша серед трьох груп
    sm2_largest = evening[
        (evening["Sub_metering_2"] > evening["Sub_metering_1"]) &
        (evening["Sub_metering_2"] > evening["Sub_metering_3"])
    ].copy()
    
    # Крок 3: розбити навпіл
    mid = len(sm2_largest) // 2
    first_half = sm2_largest.iloc[:mid]
    second_half = sm2_largest.iloc[mid:]
    
    # Кожен 3-й з першої, кожен 4-й з другої
    result = pd.concat([first_half.iloc[::3], second_half.iloc[::4]])
    
    return evening, sm2_largest, result

t = timeit.timeit(lambda: filter_evening_high_power(df), number=10)
print(f"Час виконання (10 разів): {t:.4f} с")

evening, sm2_largest, final = filter_evening_high_power(df)
print(f"\nЗаписів після 18:00 з потужністю > 6 кВт: {len(evening)}")
print(f"З них Sub_metering_2 найбільша: {len(sm2_largest)}")
print(f"Фінальна вибірка (кожен 3-й + кожен 4-й): {len(final)}")
final.head(10)

Час виконання (10 разів): 0.0350 с

Записів після 18:00 з потужністю > 6 кВт: 2882
З них Sub_metering_2 найбільша: 1061
Фінальна вибірка (кожен 3-й + кожен 4-й): 310


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime,Hour,DayOfWeek,Month
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00,18,5,12
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00,18,5,12
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00,20,3,12
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00,21,3,12
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00,21,3,12
17504,28/12/2006,21:08:00,7.352,0.000,235.45,31.2,1.0,73.0,17.0,2006-12-28 21:08:00,21,3,12
17507,28/12/2006,21:11:00,9.048,0.000,231.48,39.0,34.0,71.0,16.0,2006-12-28 21:11:00,21,3,12
17510,28/12/2006,21:14:00,9.118,0.108,231.18,39.4,36.0,72.0,16.0,2006-12-28 21:14:00,21,3,12
17513,28/12/2006,21:17:00,7.040,0.130,233.27,30.2,37.0,38.0,17.0,2006-12-28 21:17:00,21,3,12
18952,29/12/2006,21:16:00,6.146,0.116,230.53,26.6,0.0,70.0,0.0,2006-12-29 21:16:00,21,4,12


### Нормалізація та стандартизація датасету
Min-Max нормалізація та Z-score стандартизація числових атрибутів.

In [8]:
numeric_cols = ["Global_active_power", "Global_reactive_power", "Voltage",
                "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]

# Min-Max нормалізація
scaler_mm = MinMaxScaler()
df_normalized = df.copy()
df_normalized[numeric_cols] = scaler_mm.fit_transform(df[numeric_cols])

print("Min-Max нормалізація (перші 5 рядків):")
df_normalized[numeric_cols].head()

Min-Max нормалізація (перші 5 рядків):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
4,0.325005,0.379856,0.403231,0.323651,0.0,0.0125,0.548387


In [9]:
# Z-score стандартизація
scaler_std = StandardScaler()
df_standardized = df.copy()
df_standardized[numeric_cols] = scaler_std.fit_transform(df[numeric_cols])

print("Z-score стандартизація (перші 5 рядків):")
df_standardized[numeric_cols].head()

Z-score стандартизація (перші 5 рядків):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2.975591,2.629139,-1.864146,3.120054,-0.181154,-0.048773,1.262163
1,4.062977,2.789788,-2.239958,4.160250,-0.181154,-0.048773,1.143202
2,4.076284,3.343136,-2.345558,4.160250,-0.181154,0.124020,1.262163
3,4.089591,3.378836,-2.205793,4.160250,-0.181154,-0.048773,1.262163
4,2.452810,3.610885,-1.603252,2.532116,-0.181154,-0.048773,1.262163


In [10]:
print("Порівняння статистик:")
print("\nОригінал:")
print(df[numeric_cols].describe().round(4).loc[["mean", "std", "min", "max"]])
print("\nНормалізовані (Min-Max):")
print(df_normalized[numeric_cols].describe().round(4).loc[["mean", "std", "min", "max"]])
print("\nСтандартизовані (Z-score):")
print(df_standardized[numeric_cols].describe().round(4).loc[["mean", "std", "min", "max"]])

Порівняння статистик:

Оригінал:
      Global_active_power  Global_reactive_power   Voltage  Global_intensity  \
mean               1.0855                 0.1234  240.8420            4.6024   
std                1.0521                 0.1120    3.2197            4.4222   
min                0.0760                 0.0000  223.2000            0.2000   
max               11.1220                 1.3900  254.1500           48.4000   

      Sub_metering_1  Sub_metering_2  Sub_metering_3  
mean          1.1079          1.2823          6.3901  
std           6.1157          5.7873          8.4061  
min           0.0000          0.0000          0.0000  
max          88.0000         80.0000         31.0000  

Нормалізовані (Min-Max):
      Global_active_power  Global_reactive_power  Voltage  Global_intensity  \
mean               0.0914                 0.0888    0.570            0.0913   
std                0.0952                 0.0806    0.104            0.0917   
min                0.0000   

### Коефіцієнти кореляції Пірсона та Спірмена
Для двох числових атрибутів: Global_active_power та Global_intensity.

In [11]:
col1, col2 = "Global_active_power", "Global_intensity"

pearson_r, pearson_p = stats.pearsonr(df[col1], df[col2])
spearman_r, spearman_p = stats.spearmanr(df[col1], df[col2])

print(f"Кореляція між {col1} та {col2}:")
print(f"\n  Пірсон:  r = {pearson_r:.6f},  p-value = {pearson_p:.2e}")
print(f"  Спірмен: r = {spearman_r:.6f},  p-value = {spearman_p:.2e}")

t = timeit.timeit(lambda: stats.pearsonr(df[col1], df[col2]), number=10)
print(f"\nЧас Пірсона (10 разів): {t:.4f} с")

t = timeit.timeit(lambda: stats.spearmanr(df[col1], df[col2]), number=10)
print(f"Час Спірмена (10 разів): {t:.4f} с")

Кореляція між Global_active_power та Global_intensity:

  Пірсон:  r = 0.998891,  p-value = 0.00e+00
  Спірмен: r = 0.995508,  p-value = 0.00e+00

Час Пірсона (10 разів): 0.2006 с
Час Спірмена (10 разів): 2.4084 с


### One Hot Encoding категоріального атрибута
Створюємо категорію «час доби» (ніч, ранок, день, вечір) та виконуємо One Hot Encoding.

In [12]:
def time_of_day(hour):
    if 6 <= hour < 12:
        return "ранок"
    elif 12 <= hour < 18:
        return "день"
    elif 18 <= hour < 23:
        return "вечір"
    else:
        return "ніч"

df["time_of_day"] = df["Hour"].apply(time_of_day)

print("Розподіл за часом доби:")
print(df["time_of_day"].value_counts())

# One Hot Encoding
df_encoded = pd.get_dummies(df, columns=["time_of_day"], prefix="tod")

print("\nНові стовпці після One Hot Encoding:")
tod_cols = [c for c in df_encoded.columns if c.startswith("tod_")]
print(tod_cols)
df_encoded[tod_cols].head(10)

Розподіл за часом доби:
time_of_day
ніч      605220
день     518796
ранок    518760
вечір    432483
Name: count, dtype: int64

Нові стовпці після One Hot Encoding:
['tod_вечір', 'tod_день', 'tod_ніч', 'tod_ранок']


,tod_вечір,tod_день,tod_ніч,tod_ранок
0,False,True,False,False
1,False,True,False,False
2,False,True,False,False
3,False,True,False,False
4,False,True,False,False
5,False,True,False,False
6,False,True,False,False
7,False,True,False,False
8,False,True,False,False
9,False,True,False,False


### Профілювання часу основних операцій

In [ ]:
operations = {
    "Фільтр > 5 кВт": lambda: filter_high_power(df),
    "Фільтр 19-20 А + metering": lambda: filter_current_and_metering(df),
    "Випадкова вибірка 500k": lambda: random_sample_means(df),
    "Вечірній фільтр": lambda: filter_evening_high_power(df),
    "Пірсон": lambda: stats.pearsonr(df["Global_active_power"], df["Global_intensity"]),
    "Спірмен": lambda: stats.spearmanr(df["Global_active_power"], df["Global_intensity"]),
}

print("Профілювання часу виконання (середнє за 10 запусків):\n")
for name, func in operations.items():
    t = timeit.timeit(func, number=10)
    print(f"  {name:35s}: {t/10:.6f} с")

Профілювання часу виконання (середнє за 10 запусків):

  Фільтр > 5 кВт                     : 0.004347 с
  Фільтр 19-20 А + metering          : 0.004445 с
